In [2]:
import polars as pl
import glob
import json
from pathlib import Path

In [3]:
RAW_EARNINGS_PATH = Path("D:\earnings_calls\earnings_calls")
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

# Code to clean the earnings calls

In [29]:
file_pattern = DATA_DIR / "processed" / "*.parquet" 


In [30]:
# mix all files into one
df = (
    pl.scan_parquet(file_pattern,missing_columns='insert')
    # Filter only for Q&A, as Opening Statements don't fit your labeling task
    .filter(pl.col("chunk_type") == "q_and_a") 
    .collect()
)

In [31]:
df.describe()

statistic,id,chunk_id,chunk_type,question,question_speakers,answer,answer_speakers,year
str,str,f64,str,str,str,str,str,f64
"""count""","""3902104""",3.902104e6,"""3902104""","""3902104""","""3902104""","""3902104""","""3902104""",3.902104e6
"""null_count""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""",0.0
"""mean""",null,17.918452,null,null,null,null,null,2010.479875
"""std""",null,15.09163,null,null,null,null,null,5.512318
"""min""","""136899646797""",2.0,"""q_and_a""","""""","""""","""""","""""",2001.0
"""25%""",null,7.0,null,null,null,null,null,2006.0
"""50%""",null,14.0,null,null,null,null,null,2009.0
"""75%""",null,25.0,null,null,null,null,null,2016.0
"""max""","""153658080519""",201.0,"""q_and_a""","""Ørjan Kvelvane Equinor ASA - S…","""Ørjan Kvelvane Equinor ASA - S…","""Ümit Önal Türk Telekomünikasyo…","""Ümit Önal Türk Telekomünikasyo…",2020.0


In [32]:
# filter long q and lng a
filtered_df = df.filter(
    (pl.col("question").str.len_chars() > 50), 
    (pl.col("answer").str.len_chars() > 100)
)

In [33]:
filtered_df.describe()

statistic,id,chunk_id,chunk_type,question,question_speakers,answer,answer_speakers,year
str,str,f64,str,str,str,str,str,f64
"""count""","""2928233""",2.928233e6,"""2928233""","""2928233""","""2928233""","""2928233""","""2928233""",2.928233e6
"""null_count""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""",0.0
"""mean""",null,17.458403,null,null,null,null,null,2011.085515
"""std""",null,14.489465,null,null,null,null,null,5.686741
"""min""","""136899658969""",2.0,"""q_and_a""","""""Sandy"" Allan The Coca-Cola Co…","""""Sandy"" Allan The Coca-Cola Co…","""A .P. Chen D-Link Corporation …","""A .P. Chen D-Link Corporation …",2001.0
"""25%""",null,7.0,null,null,null,null,null,2006.0
"""50%""",null,13.0,null,null,null,null,null,2010.0
"""75%""",null,24.0,null,null,null,null,null,2017.0
"""max""","""153658080519""",201.0,"""q_and_a""","""Ørjan Kvelvane Equinor ASA - S…","""Ørjan Kvelvane Equinor ASA - S…","""Ümit Önal Türk Telekomünikasyo…","""Ümit Önal Türk Telekomünikasyo…",2020.0


In [34]:
# make a unique id
filtered_df = filtered_df.with_columns(
    pl.concat_str(["year","id", "chunk_id"], separator="_").alias("unique_id")
)

In [35]:
final_df = filtered_df.drop(["id","chunk_id"])
final_df = final_df.rename({"unique_id": "id"})

In [41]:
unique_calls_df = final_df.unique(subset=["id"], keep="first")

In [ ]:
sample_size = min(500, len(unique_calls_df))
sampled_df = unique_calls_df.sample(n=sample_size, with_replacement=False, seed=42)

final_df = (
    sampled_df
    .drop("id") 
    .with_row_index("id", offset=1)
    .with_columns(pl.lit("").alias("label"))
    .select(["id", "question", "answer", "label"])
)

json_data = final_df.to_dicts()

output_filename = DATA_DIR / "processed" / "uncertainty_labeling_sample.json"
with open(output_filename, "w") as f:
    json.dump(json_data, f, indent=2)

DuplicateError: column with name 'id' has more than one occurrence